# 🌲 Step 2 — Cloud-Proof Masking & S1/S2 Fusion
**Forest Carbon Continuity Pipeline · Sentinel-1/2 Hybrid for EU LULUCF Compliance**

## What this step does
Generates a synthetic co-registered Sentinel-1 (VH/VV radar) + Sentinel-2 (NIR/Red optical) scene
with a peat/water block and 25% cloud cover, then builds the forest mask the pipeline actually
uses: a **SAR rule** gated by clear-sky NDVI. It quantifies how cloud alone destroys optical-only
mapping while SAR keeps working.

## The Science
Radar (SAR) sends its own microwave signal, so **clouds are invisible to it**. Two physics rules
separate forest from peat/water in Sentinel-1:
- **VH backscatter in dB**: `10·log10(VH)`. Forest canopies scatter strongly (>-15 dB); smooth
  water/peat is dark.
- **VH/VV ratio**: canopy depolarises the radar pulse (volume scattering), so VH grows relative
  to VV (ratio > 0.15 over forest; water stays low).

The fused product is `SAR mask AND (NDVI > 0.45)` — SAR provides the cloud-proof base, optical
NDVI trims false positives on day of clear sight.

In [ ]:
# ============================================================
# Environment detection & package setup (runs first, auto-detects
# Google Colab vs local Jupyter - nothing needs manual pip install)
# ============================================================
import sys, subprocess

def _ensure(pkg, import_name=None):
    import_name = import_name or pkg
    try:
        __import__(import_name)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    colab_files = None

_ensure("numpy"); _ensure("pandas"); _ensure("matplotlib")
_ensure("rasterio"); _ensure("geopandas"); _ensure("scikit-learn")
_ensure("openpyxl"); _ensure("ipywidgets")

def outpath(name):
    """Returns a working save path for the current environment."""
    return f"/content/{name}" if IN_COLAB else name

print(f"Environment detected: {'Google Colab' if IN_COLAB else 'Local Jupyter'}")

# ============================================================
# Unified cross-environment helpers:
# download buttons, regenerate buttons, upload widgets
# ============================================================
import zipfile
import ipywidgets as widgets
from IPython.display import display, FileLink, clear_output

_generated_files = {}  # friendly label -> filepath, used by the final "Download All" bundle

def register_file(label, path):
    if label:
        _generated_files[label] = path

def make_download_button(path, label=None, register_as=None):
    """Cross-environment download button. Colab: real browser download.
    Local Jupyter: clickable FileLink. Reads the file fresh at click time."""
    label = label or f"Download {path}"
    if register_as:
        register_file(register_as, path)
    btn = widgets.Button(description=label, button_style="info", icon="download")
    out = widgets.Output()

    def on_click():
        with out:
            clear_output(wait=True)
            if IN_COLAB:
                colab_files.download(path)
            else:
                display(FileLink(path))
                print(f"Click the link above to download '{path}'.")

    btn.on_click(on_click)
    display(widgets.VBox([btn, out]))

def make_regenerate_button(rerun_fn, label="🔄 Regenerate & Re-run"):
    """Button that calls rerun_fn() (fresh random seed, full re-run of this
    section) and shows the refreshed output inline below the button."""
    btn = widgets.Button(description=label, button_style="warning", icon="refresh")
    out = widgets.Output()

    def on_click():
        with out:
            clear_output(wait=True)
            rerun_fn()

    btn.on_click(on_click)
    display(widgets.VBox([btn, out]))

def make_upload_widget(on_loaded, accept=".csv,.xlsx", label="Upload replacement file"):
    """Colab: native blocking file picker via google.colab.files.upload().
    Local Jupyter: ipywidgets.FileUpload + Load button (non-blocking)."""
    out = widgets.Output()

    if IN_COLAB:
        btn = widgets.Button(description=label, button_style="primary", icon="upload")

        def on_click():
            with out:
                clear_output(wait=True)
                print("Choose a file in the dialog, or Cancel to keep current data.")
                uploaded = colab_files.upload()
                if uploaded:
                    fname = next(iter(uploaded))
                    on_loaded(fname, uploaded[fname])
                else:
                    print("No file selected - keeping current data.")

        btn.on_click(on_click)
        display(widgets.VBox([btn, out]))
    else:
        uploader = widgets.FileUpload(accept=accept, multiple=False, description="Choose file")
        load_btn = widgets.Button(description=label, button_style="primary", icon="upload")

        def on_click():
            with out:
                clear_output(wait=True)
                val = uploader.value
                if not val:
                    print("No file selected yet - choose a file above, then click the button.")
                    return
                item = list(val.values())[0] if isinstance(val, dict) else val[0]
                if isinstance(item, dict) and "metadata" in item:
                    fname = item["metadata"]["name"]     # ipywidgets 7.x shape
                else:
                    fname = item["name"]                  # ipywidgets 8.x shape
                on_loaded(fname, bytes(item["content"]))

        load_btn.on_click(on_click)
        display(widgets.VBox([uploader, load_btn, out]))

def make_download_all_button(label="⬇️ Download ALL outputs (.zip)"):
    """Zips every output registered so far and offers it as one download."""
    zip_path = outpath("outputs_bundle.zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for _label, _path in _generated_files.items():
            try:
                zf.write(_path)
            except FileNotFoundError:
                pass
    make_download_button(zip_path, label=label)

print("Helpers ready: make_download_button / make_regenerate_button / make_upload_widget")

In [ ]:
# ============================================================
# Standalone synthetic S1 + S2 scene (Day-1 ready)
# ============================================================
import numpy as np
import rasterio
from rasterio.transform import from_bounds

H, W = 120, 120
BOUNDS = (-6.4, 52.95, -6.1, 53.15)
TRANSFORM = from_bounds(*BOUNDS, W, H)

def generate_s1_s2_scene(seed=7):
    """Synthetic S1 (VH/VV linear power) + S2 (NIR/Red) with peat block and 25% cloud."""
    rng = np.random.default_rng(seed)
    vh = rng.uniform(0.01, 0.15, (H, W)).astype("float32")
    vv = rng.uniform(0.05, 0.25, (H, W)).astype("float32")
    vh[45:75, 45:75] = 0.001; vv[45:75, 45:75] = 0.30   # peat/water: dark VH, bright VV
    nir = np.clip(0.55 + rng.normal(0, 0.08, (H, W)), 0.02, 0.95).astype("float32")
    red = np.clip(0.18 + rng.normal(0, 0.05, (H, W)), 0.01, 0.60).astype("float32")
    cloud = rng.random((H, W)) < 0.25
    nir_c, red_c = nir.copy(), red.copy()
    nir_c[cloud] = np.clip(nir_c[cloud] * 0.4 + 0.35, 0, 1)   # haze pushes NDVI toward 0
    red_c[cloud] = np.clip(red_c[cloud] * 1.6 + 0.15, 0, 1)
    return vh, vv, nir, red, nir_c, red_c, cloud

vh, vv, nir, red, nir_c, red_c, cloud = generate_s1_s2_scene()
print(f"Synthetic scene: {H}x{W} pixels, {cloud.mean()*100:.0f}% under cloud")

In [ ]:
# ============================================================
# Run the fusion: SAR forest mask + NDVI gate + cloud comparison
# ============================================================
import pandas as pd
import matplotlib.pyplot as plt

PIXEL_AREA_HA = 100.0 / 10000.0   # 10 m pixels -> 0.01 ha each

def run_fusion(vh, vv, nir_clean, red_clean, nir_cloudy, red_cloudy, tag="synthetic"):
    with np.errstate(divide="ignore"):
        vh_db = 10 * np.log10(np.where(vh > 0, vh, 1e-10))
    ratio = np.divide(vh, vv, out=np.zeros_like(vh), where=vv != 0)
    sar_mask = ((vh_db > -15) & (ratio > 0.15)).astype("uint8")

    def ndvi(nir_, red_):
        num = nir_ - red_
        den = nir_ + red_
        return np.divide(num, den, out=np.zeros_like(nir_), where=den != 0)

    ndvi_clean = ndvi(nir_clean, red_clean)
    ndvi_cloudy = ndvi(nir_cloudy, red_cloudy)
    optical_clean = ndvi_clean > 0.45
    optical_cloudy = ndvi_cloudy > 0.45
    fused = sar_mask.astype(bool) & optical_clean

    stats_df = pd.DataFrame({
        "Scenario": ["Optical only - clear sky", "Optical only - 25% cloud",
                     "SAR only (Sentinel-1)", "Fused S1 + S2"],
        "Forest pixels": [int(optical_clean.sum()), int(optical_cloudy.sum()),
                          int(sar_mask.sum()), int(fused.sum())],
    })
    stats_df["Forest area (ha)"] = (stats_df["Forest pixels"] * PIXEL_AREA_HA).round(1)
    print(stats_df.to_string(index=False))

    fig, axes = plt.subplots(2, 2, figsize=(11, 8.5))
    im0 = axes[0, 0].imshow(vh_db, cmap="gray", vmin=-30, vmax=-5)
    axes[0, 0].set_title("Sentinel-1 VH (dB)"); plt.colorbar(im0, ax=axes[0, 0], fraction=0.046)
    axes[0, 1].imshow(sar_mask, cmap="Greens")
    axes[0, 1].set_title("SAR forest mask (1 = forest)")
    im2 = axes[1, 0].imshow(ndvi_cloudy, cmap="RdYlGn", vmin=-0.2, vmax=0.9)
    axes[1, 0].set_title("Sentinel-2 NDVI (cloudy)"); plt.colorbar(im2, ax=axes[1, 0], fraction=0.046)
    axes[1, 1].imshow(fused, cmap="Greens")
    axes[1, 1].set_title("Fused S1+S2 forest (SAR base + NDVI gate)")
    for ax in axes.flat:
        ax.set_axis_off()
    plt.tight_layout()

    chart_path = outpath("step2_fusion_chart.png")
    fig.savefig(chart_path, dpi=120, bbox_inches="tight")
    plt.show()

    mask_path = outpath("step2_forest_mask.tif")
    with rasterio.open(mask_path, "w", driver="GTiff", height=H, width=W, count=1,
                       dtype="uint8", crs="EPSG:4326", transform=TRANSFORM) as dst:
        dst.write(sar_mask, 1)

    stats_path = outpath("step2_fusion_stats.xlsx")
    stats_df.to_excel(stats_path, index=False)

    make_download_button(chart_path, label="⬇️ Download fusion chart (.png)",
                         register_as=f"Step 2 chart ({tag})")
    make_download_button(mask_path, label="⬇️ Download forest mask (.tif)",
                         register_as=f"Step 2 forest mask ({tag})")
    make_download_button(stats_path, label="⬇️ Download comparison stats (.xlsx)",
                         register_as=f"Step 2 stats ({tag})")
    return stats_df

stats_df = run_fusion(vh, vv, nir, red, nir_c, red_c)

**Interpretation:** look at the "Forest area" column. Under 25% simulated cloud, the
**optical-only** estimate collapses (haze drags NDVI below the 0.45 gate), while the **SAR mask**
barely moves — radar does not care about clouds. The fused product is the conservative
intersection: SAR finds the forest, clear-sky NDVI confirms it. On overcast Atlantic Ireland,
that difference is the difference between a usable carbon estimate and a hole-filled guess.

In [ ]:
# ============================================================
# Optional: upload your own Sentinel-1 VH/VV stack
# (2-band GeoTIFF: band 1 = VH, band 2 = VV, linear power, same grid as the S2 pair)
# ============================================================
def handle_s1_upload(fname, content_bytes):
    tmp = outpath("uploaded_s1_stack.tif")
    open(tmp, "wb").write(content_bytes)
    try:
        with rasterio.open(tmp) as src:
            if src.count < 2:
                print(f"❌ Upload rejected - '{fname}' has {src.count} band(s); need 2 (VH, VV).")
                return
            u_vh = src.read(1).astype("float32")
            u_vv = src.read(2).astype("float32")
    except Exception as e:
        print(f"❌ Could not read '{fname}': {e}")
        return
    if u_vh.shape != nir.shape:
        print(f"❌ Upload rejected - shape {u_vh.shape} must match the optical grid {nir.shape}.")
        return
    print(f"✅ SUCCESS: loaded VH/VV from '{fname}'. Re-running fusion with your radar data...")
    run_fusion(u_vh, u_vv, nir, red, nir_c, red_c, tag=f"uploaded:{fname}")

make_upload_widget(handle_s1_upload, accept=".tif,.tiff", label="Upload VH/VV stack (.tif)")
print("Skip to continue with the synthetic scene.")

# ============================================================
# Regenerate control
# ============================================================
def rerun_step2(seed=None):
    """Fresh synthetic scene with a new seed + full fusion re-run."""
    global vh, vv, nir, red, nir_c, red_c, cloud
    seed = seed or np.random.randint(0, 10_000)
    vh, vv, nir, red, nir_c, red_c, cloud = generate_s1_s2_scene(seed=seed)
    print(f"[seed={seed}] fresh synthetic scene, {cloud.mean()*100:.0f}% cloud")
    run_fusion(vh, vv, nir, red, nir_c, red_c, tag=f"seed-{seed}")

make_regenerate_button(rerun_step2, label="🔄 Regenerate scene & re-run fusion")
make_download_all_button()